# Sistemas RAG (Retrieval-Augmented Generation)

En este notebook vamos a construir un sistema **RAG** desde cero, combinando dos herramientas que ya habeis visto:

1. **Modelo de lenguaje** (Phi-2) para generar respuestas.
2. **Embeddings + similitud del coseno** para recuperar información relevante basado en una colección de información que añadiremos.

## ¿Qué es RAG?

Los modelos de lenguaje como GPT o Phi-2 tienen un conocimiento limitado a sus datos de entrenamiento y pueden generar información incorrecta (alucinaciones). RAG soluciona esto recuperando documentos relevantes de una base de conocimiento externa y **añadiéndolos como contexto** al prompt del modelo. Así la respuesta se basa en información actualizada y verificable.

El flujo típico es:
1. El usuario hace una pregunta.
2. Buscamos en nuestra base de documentos los más similares a la pregunta (usando embeddings).
3. Construimos un prompt con los documentos recuperados y la pregunta.
4. El modelo de lenguaje genera una respuesta basada en ese contexto.

## 1. Cargamos las librerías y los modelos necesarios

Usaremos el mismo modelo de lenguaje (`microsoft/Phi-2`) y el mismo modelo de embeddings (`all-mpnet-base-v2`) que ya conocéis.

In [ ]:
%pip install -U torch transformers sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 27.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
# --- Modelo de lenguaje ---
model_name_llm = "microsoft/Phi-2"
cache_dir = "./model_cache"

print("Cargando tokenizador y modelo de lenguaje...")
tokenizer = AutoTokenizer.from_pretrained(model_name_llm, cache_dir=cache_dir)
model_llm = AutoModelForCausalLM.from_pretrained(model_name_llm, cache_dir=cache_dir)


# --- Modelo de embeddings ---
print("Cargando modelo de embeddings...")
model_emb = SentenceTransformer("all-mpnet-base-v2")

print("¡Modelos listos!")

Cargando tokenizador y modelo de lenguaje...


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


¡Modelos listos!


## 2. Creamos un corpus de datos

Usaremos un archivo json con pares clave-valor como base de conocimiento.
Ojo! En un sistema RAG real, estos archivos y documentos provendrían de una base de datos, archivos PDF, páginas web, etc. Aquí usaremos un json con datos sobre atención al cliente (simulando FAQs).

In [ ]:
ruta_json = "documentos.json"   # Cambia por la ruta de tu archivo

with open(ruta_json, "r", encoding="utf-8") as f:
    datos = json.load(f)

# Extraemos los textos: si es un dict, tomamos los valores; si es lista, tomamos el campo "texto"
if isinstance(datos, dict):
    documentos = list(datos.values())
    ids = list(datos.keys())
else:
    # Suponemos que cada elemento tiene un campo "texto"
    documentos = [item["texto"] for item in datos]
    ids = [item.get("id", str(i)) for i, item in enumerate(datos)]

print(f"Se cargaron {len(documentos)} documentos.")
print("Algunos documentos:", documentos[0:5], "")

Se cargaron 5 documentos.
Algunos documentos: ['The Hospital is at Oliveira Street 12', 'We will not provide speciallized treatments without an appointment', 'We are open from 9am to 5pm', 'We will provide only one free consultation', 'This hospital is called the Oliveira Hospital'] 


## 3. Función para recuperar documentos relevantes

Dada una consulta, calculamos su embedding y el de todos los documentos. Luego medimos la similitud del coseno y devolvemos los documentos con mayor puntuación (top-k).

In [ ]:
def recuperar_documentos(consulta, documentos, modelo_emb, top_k=5, umbral=0.5):
    """
    Devuelve los documentos más relevantes para la consulta.
    Si ninguna similitud supera el umbral, devuelve lista vacía.
    """
    # Crear embeddings
    vec_consulta = modelo_emb.encode([consulta])
    vec_docs = modelo_emb.encode(documentos)

    # Calcular similitudes
    similitudes = cosine_similarity(vec_consulta, vec_docs)[0]

    # Ordenar por similitud (mayor a menor)
    indices_ordenados = np.argsort(similitudes)[::-1]

    # Filtrar por umbral
    relevantes = []
    for idx in indices_ordenados:
        if similitudes[idx] >= umbral:
            relevantes.append((documentos[idx], similitudes[idx]))
        else:
            break  # como están ordenados, los siguientes serán menores

    return relevantes[:top_k]

# Probemos con una consulta
consulta_prueba = "Where can i find the hospital?"
resultados = recuperar_documentos(consulta_prueba, documentos, model_emb, top_k=2)
print("Consulta:", consulta_prueba)
print("Documentos relevantes:")
for doc, sim in resultados:
    print(f"- {doc} (similitud: {sim:.4f})")

Consulta: Where can i find the hospital?
Documentos relevantes:
- The Hospital is at Oliveira Street 12 (similitud: 0.6937)
- This hospital is called the Oliveira Hospital (similitud: 0.6218)


## 4. Función para generar respuesta con contexto RAG

Ahora unimos la recuperación con el modelo de lenguaje. Construimos un prompt que incluya los documentos recuperados y luego llamamos al modelo.

In [ ]:
def generar_respuesta_rag(consulta, documentos_recuperados):
    """
    Genera una respuesta usando el modelo de lenguaje,
    añadiendo los documentos recuperados como contexto.
    """
    if not documentos_recuperados:
        contexto = "No hay información relevante disponible."
    else:
        # Unimos los documentos en un solo texto
        contexto = " ".join([doc for doc, _ in documentos_recuperados])

    prompt = (
        "Answer the question based only on the context provided\n"
        f"Context: {contexto}\n"
        f"Question: {consulta}\n"
        "Answer:"
    )

    # Tokenizar y generar
    tokens = tokenizer(prompt, return_tensors="pt")
    output = model_llm.generate(
        tokens["input_ids"],
        max_new_tokens=50,          # longitud máxima de la respuesta
        temperature=0.2,
        do_sample=False,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )
    respuesta_completa = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extraer solo la parte después de "Respuesta:"
    if "Respuesta:" in respuesta_completa:
        respuesta = respuesta_completa.split("Respuesta:")[-1].strip()
    else:
        respuesta = respuesta_completa

    return respuesta, contexto

## 5. Probemos el sistema RAG completo

Ahora podemos interactuar: escribe una pregunta y verás qué documentos recupera y qué respuesta genera.

In [ ]:
# Pedir pregunta al usuario
consulta_usuario = input("Write your question: ")

# 1. Recuperar documentos relevantes
docs_relevantes = recuperar_documentos(consulta_usuario, documentos, model_emb, top_k=2, umbral=0.4)

print("\nDocumentos recuperados:")
if docs_relevantes:
    for doc, sim in docs_relevantes:
        print(f"- {doc} (similitud: {sim:.4f})")
else:
    print("Ningún documento supera el umbral de similitud.")

# 2. Generar respuesta con contexto
respuesta, contexto_usado = generar_respuesta_rag(consulta_usuario, docs_relevantes)

print("\n--- Respuesta generada ---")
print(respuesta)
print("\n(Contexto usado:", contexto_usado, ")")

Write your question: Where is the hospital?


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Documentos recuperados:
- The Hospital is at Oliveira Street 12 (similitud: 0.6550)
- This hospital is called the Oliveira Hospital (similitud: 0.6006)

--- Respuesta generada ---
Answer the question based only on the context provided
Context: The Hospital is at Oliveira Street 12 This hospital is called the Oliveira Hospital
Question: Where is the hospital?
Answer: The hospital is at Oliveira Street 12.


(Contexto usado: The Hospital is at Oliveira Street 12 This hospital is called the Oliveira Hospital )


## 6. Comparación: sin contexto vs con contexto

Para apreciar la diferencia, generaremos una respuesta sin recuperar documentos, usando solo el modelo (como en el código original). Así veremos cómo el contexto RAG mejora la precisión.

In [ ]:
def generar_sin_contexto(consulta):
    """Usa el modelo de lenguaje sin contexto adicional."""
    prompt = (
        "Answer the next question\n"
        f"Question: {consulta}\n"
        "Answer:"
    )
    tokens = tokenizer(prompt, return_tensors="pt")
    output = model_llm.generate(
        tokens["input_ids"],
        max_new_tokens=50,
        temperature=0.2,
        do_sample=False,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )
    respuesta_completa = tokenizer.decode(output[0], skip_special_tokens=True)
    if "Respuesta:" in respuesta_completa:
        return respuesta_completa.split("Respuesta:")[-1].strip()
    else:
        return respuesta_completa

# Probemos con una pregunta específica
pregunta_ejemplo = "Where is the hospital"

print("Pregunta:", pregunta_ejemplo)
print("\n--- Respuesta SIN contexto ---")
print(generar_sin_contexto(pregunta_ejemplo))

# Recuperamos documentos para la misma pregunta
docs = recuperar_documentos(pregunta_ejemplo, documentos, model_emb, top_k=2)
print("\nDocumentos recuperados para RAG:")
for d, s in docs:
    print(f"- {d} (sim: {s:.4f})")

resp_rag, ctx = generar_respuesta_rag(pregunta_ejemplo, docs)
print("\n--- Respuesta CON contexto (RAG) ---")
print(resp_rag)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Pregunta: Where is the hospital

--- Respuesta SIN contexto ---
Answer the next question
Question: Where is the hospital
Answer: The hospital is located in the city.



The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Documentos recuperados para RAG:
- The Hospital is at Oliveira Street 12 (sim: 0.6603)
- This hospital is called the Oliveira Hospital (sim: 0.6443)

--- Respuesta CON contexto (RAG) ---
Answer the question based only on the context provided
Context: The Hospital is at Oliveira Street 12 This hospital is called the Oliveira Hospital
Question: Where is the hospital
Answer: The hospital is at Oliveira Street 12.



## 7. Conclusiones

- Hemos construido un sistema RAG muy simple pero funcional.
- La recuperación permite que el modelo acceda a información que no está en sus parámetros, reduciendo alucinaciones.
- En este ejemplo, el modelo con contexto se limita a lo que hemos proporcionado, pero esto no significa que con esto evitemos alucinaciones!.
- En un sistema real, usarías una base de datos vectorial (como Chroma, FAISS) para manejar miles de documentos y recuperar los más relevantes de forma eficiente.

### Ampliaciones posibles
- Probar con otros modelos de lenguaje.
- Ajustar el umbral de similitud y el número de documentos recuperados.
- Hacer que lea .csv .md .txt .pdf y otros tipos de archivos.

## Reto de práctica
- Descarga un json con datos de libros/películas/series/animes/etc en [kaggle.com](https://kaggle.com) y pregunta al chatbot para ver si puedes recuperar información sobre ellas.


## Práctica final de RA

En la próxima clase veremos cómo transformar esto en una API que permita a cualquier usuario hacer preguntas y ver respuestas relevantes según nuestros documentos.